In [1]:
chr(0)

'\x00'

(a) It returns \x00

In [2]:
repr(chr(0))

"'\\x00'"

(b) The string representation is '\\x00', which has an additional \ and is enclosed with single quotes

In [3]:
chr(0)

'\x00'

In [4]:
print(chr(0))

 


In [5]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [6]:
print("this is a test" + chr(0) + "string")

this is a test string


(c) when using ()print, the character does not get printed to the output - we have empty print. Without print(), the character is represented as \x00

In [7]:
test_string = "hello! こんにちは！"
utf8_encoded = test_string.encode('utf-8')
print(utf8_encoded)
print(type(utf8_encoded))
list(utf8_encoded)
print(len(test_string))
print(len(utf8_encoded))
print(utf8_encoded.decode('utf-8'))

b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf\xef\xbc\x81'
<class 'bytes'>
13
25
hello! こんにちは！


In [8]:
test_string = "hello! こんにちは！"
utf8_encoded = test_string.encode('utf-16')
print(utf8_encoded)
print(type(utf8_encoded))
list(utf8_encoded)
print(len(test_string))
print(len(utf8_encoded))
print(utf8_encoded.decode('utf-16'))

b'\xff\xfeh\x00e\x00l\x00l\x00o\x00!\x00 \x00S0\x930k0a0o0\x01\xff'
<class 'bytes'>
13
28
hello! こんにちは！


In [9]:
test_string = "hello! こんにちは！"
utf8_encoded = test_string.encode('utf-32')
print(utf8_encoded)
print(type(utf8_encoded))
list(utf8_encoded)
print(len(test_string))
print(len(utf8_encoded))
print(utf8_encoded.decode('utf-32'))

b'\xff\xfe\x00\x00h\x00\x00\x00e\x00\x00\x00l\x00\x00\x00l\x00\x00\x00o\x00\x00\x00!\x00\x00\x00 \x00\x00\x00S0\x00\x00\x930\x00\x00k0\x00\x00a0\x00\x00o0\x00\x00\x01\xff\x00\x00'
<class 'bytes'>
13
56
hello! こんにちは！


2.(a)
We can see that in utf-16 and utf-32, there are a lot of "padding" bytes \x00, which we saw earlier does not represent actual characters. They occur a lot but do not carry meaningful semantic information. The high frequency of their occurence can cause them to take a major part in the tokenizer dictionary, and making it hard to merge those byte combinations with actual meaningful information, leading to inefficient tokenization.



In [20]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join(bytes([b]).decode('utf-8') for b in bytestring)


decode_utf8_bytes_to_str_wrong("你好".encode('utf-8'))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe4 in position 0: unexpected end of data

2.(b)

The above example produces incorrect output. The function incorrectly assumes each utf8 encoded byte corresponding to a character and tries to directly decode it. When then encoded character corresponds to more than 1 byte, the function produces incorrect output.

In [44]:
b = bytes([0xD8, 0xD8])

In [45]:
b.decode('utf-8')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd8 in position 0: invalid continuation byte

2.(c)

The byte sequence 0xC0, 0x37 cannot be decoded to any Unicode characters because in UTF-8, the byte must follow certain pattern (e.g. the second byte starts with a continuation byte) to allow a low-bit representation of unicode. With 0xD8 0xD8, it is expected that the second byte starts with a continuation byte and shall give an error when it does not.

(train_bpe_tinystories)

(a)
The training took a total of 2 minutes and a peak memory of 621MB. The longest token in the vocab is " accomplishment" (2 other tokens share the same length as it), and it does make sense.

(b)
The step that took the most time was the pretoknization step, specifically, when matching for the pretokenization regex and encoding them into utf-8 bytes tuples. This specific line of code took 78% of my total runtime.

(train_bpe_expts_owt)
(a) The longest token in the vocabulary is '----------------------------------------------------------------', which is a sequence of 64 '-'

(b) Common short sequence like ' a', ' t' appear very early in the trained tokenizer for both datasets. Yet OpenWebText contains many more tokens that consist of unprintable chars, while the tokenizer of TinyStories almost only consists of normal English characters.

(tokenizer_experiments)

(a)
The compression ratio of TinyStories is around 3.98 bytes / token. The compression ratio of the OpenWebText tokenizer is around 

In [4]:
import sys
sys.path.insert(0, "/Users/paulzhu/learn/assignment1-basics")
from answer.transformers.transformer_lm import TransformerLM


llm = TransformerLM(
    d_model=1600,
    num_heads=25,
    d_ff=6400,
    vocab_size=50257,
    context_length=1024,
    num_layers=48)

print(f"Number of parameters: {sum(p.numel() for p in llm.parameters()):,}")

Number of parameters: 2,127,057,600


(transformer_accounting)

(a)
The number of parameters can be decomposed into the following parts:

- TransformerLM
    - Emebedding layer: vocab_size * d_model = 50257 * 1600 = 80411200
    - Transformer Block:
        - MHA:
            - down projection: 3 * num_heads * d_k * d_model = 3 * 25 * 64 * 1600 = 7680000
            - up projection: d_model * num_heads * d_v = 1600 * 25 * 64 = 2560000
            - total = 10240000
        - SwiGLU:
            - 3 Linear Layers: 3 * d_model * d_ff = 3 * 1600 * 6400 = 30720000
        - 2 * RMSNorm:
            - 2 * d_model = 3200
        - Total (48 layers):
            - (10240000 + 30720000 + 3200) * 48 = 1966233600
    - RMSNorm
        - 1600
    - Linear
        - d_model * vocab_size = 1600 * 50257 = 80411200
    - Total = 80411200 + 1966233600 + 1600 + 80411200 = 2127057600 (2.1B)

Using FP16, the memory required to load the model weights is 2127057600 * 4 bytes = 8508230400 bytes (~8GB)


(b)

The matrix multiplicaiton requires in each individual components are:

- Embedding layer: None
- Transformers block
    - RMSNorm: None
    - Multi-head attention:
        - down projection $xW_{proj}^T$
            - $x\in (B, T, 3, 1, d_{model}), W^T\in (3, d_{model}, d_k * num_heads)$
            - FLOP = 2 * 1 * 1600 * 64 * 25 * 3 * 1024 = 15,728,640,000
        - Scaled product attention:
            - $QK^T$
                - $Q\in (B, h, T, d_k)$
                - $K^T\in (B, h, d_k, T)$
                - FLOP = 2 * 1024 * 64 * 1024 * 25 = 3,355,443,200
            - $\text{weight}\cdot V$
                - weight$\in (B, h, T, T)$
                - $V\in (B, h, T, d_v)$
                - FLOP = 2 * 1024 * 1024 * 64 * 25 = 3,355,443,200
        - up projection $\text{attn}\cdot W_O^T$
            - weight$\in (B, T, h*d_v)$
            - $W_O^T\in (h*d_v, d_{model})$
            - FLOP = 2 * 1024 * 1600 * 1600 = 5,242,880,000
        - FLOP = 15728640000 + 3355443200 + 3355443200 + 5242880000 = 27,682,406,400
    - FFN:
        - $(\text{SiLU}(xW_1^T)\odot (xW_2^T))W_2^T$
            - $x\in (B, T, d)$
            - $W_1^T\in (d_{model}, d_{ff})$
            - FLOP = 2 * 1024 * 1600 * 6400 = 20971520000
            - $x\in (B, T, d)$
            - $W_3^T\in (d_{model}, d_{ff})$
            - FLOP = 2 * 1024 * 1600 * 6400 = 20971520000
            - $x\in (B, T, d)$
            - $W_2^T\in (d_{ff}, d_{model})
            - FLOP = 2 * 1024 * 6400 * 1600  = 20971520000
        - FLOP = 3 * 20971520000 = 62,914,560,000
    - FLOP = 27682406400 + 62914560000 = 90,596,966,400
    - FLOP (48 layers) = 90596966400 * 48 = 4,348,654,387,200
- Linear layer $xW^T$
    - $x\in (B, T, d_{model})$
    - $W^T\in (d, \text{vocab\_size})$
    - FLOP = 2 * 1024 * 1600 * 50257 = 164,682,137,600
- FLOP = 4348654387200 + 164682137600 = 4,513,336,524,800


(c)
Looking at the analysis above, we can see that the FFN in each transformer block contributed 62B flop, multiplying by 48 layers, we find that all FFN layers contributed 2.9T FLOPs in total, accounting for the most FLOPs in this LLM architecture.


(d)

Using the same calculation above we can get the FLOP calculation result for the following models:


GPT-2 small:
Per Transformer Layer:
  Attention:
    - Down projection (QKV): 3,623,878,656
    - QK^T: 1,610,612,736
    - Weighted V: 1,610,612,736
    - Up projection: 1,207,959,552
    - Total Attention: 8,053,063,680
  FFN (SwiGLU):
    - xW_1^T: 10,066,329,600
    - xW_3^T: 10,066,329,600
    - result*W_2^T: 10,066,329,600
    - Total FFN: 30,198,988,800
  Total per layer: 38,252,052,480

All Transformer Blocks (12 layers):
  459,024,629,760

Final Linear Layer:
  79,047,426,048

TOTAL FLOPs: 538,072,055,808

GPT-2 medium:
Per Transformer Layer:
  Attention:
    - Down projection (QKV): 6,442,450,944
    - QK^T: 2,147,483,648
    - Weighted V: 2,147,483,648
    - Up projection: 2,147,483,648
    - Total Attention: 12,884,901,888
  FFN (SwiGLU):
    - xW_1^T: 13,421,772,800
    - xW_3^T: 13,421,772,800
    - result*W_2^T: 13,421,772,800
    - Total FFN: 40,265,318,400
  Total per layer: 53,150,220,288

All Transformer Blocks (24 layers):
  1,275,605,286,912

Final Linear Layer:
  105,396,568,064

TOTAL FLOPs: 1,381,001,854,976


GPT-2 large:
Per Transformer Layer:
  Attention:
    - Down projection (QKV): 10,066,329,600
    - QK^T: 2,684,354,560
    - Weighted V: 2,684,354,560
    - Up projection: 3,355,443,200
    - Total Attention: 18,790,481,920
  FFN (SwiGLU):
    - xW_1^T: 16,777,216,000
    - xW_3^T: 16,777,216,000
    - result*W_2^T: 16,777,216,000
    - Total FFN: 50,331,648,000
  Total per layer: 69,122,129,920

All Transformer Blocks (36 layers):
  2,488,396,677,120

Final Linear Layer:
  131,745,710,080

TOTAL FLOPs: 2,620,142,387,200


GPT-2 XL:
Per Transformer Layer:
  Attention:
    - Down projection (QKV): 15,728,640,000
    - QK^T: 3,355,443,200
    - Weighted V: 3,355,443,200
    - Up projection: 5,242,880,000
    - Total Attention: 27,682,406,400
  FFN (SwiGLU):
    - xW_1^T: 20,971,520,000
    - xW_3^T: 20,971,520,000
    - result*W_2^T: 20,971,520,000
    - Total FFN: 62,914,560,000
  Total per layer: 90,596,966,400

All Transformer Blocks (48 layers):
  4,348,654,387,200

Final Linear Layer:
  164,682,137,600

TOTAL FLOPs: 4,513,336,524,800


---
If we look at the Attention part of each model, the number of FLOP from smaller to larger models are:
8,053,063,680 -> 12,884,901,888 -> 18,790,481,920 -> 27,682,406,400

While for FFN, the FLOPs are:
30,198,988,800 -> 40,265,318,400 -> 50,331,648,000 -> 62,914,560,000

For the final linear layer:
79,047,426,048 -> 105,396,568,064 -> 131,745,710,080 -> 164,682,137,600

We can see that eahc model config changes from smaller to larger model brings about 50% increase in attention FLOP, while less than 50% change in FFN FLOPs (1/3 to 1/4 to 1/5). The final linear projection layer observes the same trend of change as the per-layer FFN, so we can conclude that the model config change brings more changes in the attention part for the given set of GPT2 configs.


(e)

GPT-2 XL (long context)

Per Transformer Layer:
  Attention:
    - Down projection (QKV): 251,658,240,000
    - QK^T: 858,993,459,200
    - Weighted V: 858,993,459,200
    - Up projection: 83,886,080,000
    - Total Attention: 2,053,531,238,400
  FFN (SwiGLU):
    - xW_1^T: 335,544,320,000
    - xW_3^T: 335,544,320,000
    - result*W_2^T: 335,544,320,000
    - Total FFN: 1,006,632,960,000
  Total per layer: 3,060,164,198,400

All Transformer Blocks (48 layers):
  146,887,881,523,200

Final Linear Layer:
  2,634,914,201,600

TOTAL FLOPs: 149,522,795,724,800



We can see that when increasing the context length by 16x, all linear layers (QKV down projection, up projection, FFN, final linear projection) has the same increase of 16x in FLOP. However, the attention calculation including QK^T and weighted V calculation are increased by 256x, causing the FLOP increase in attention to be much larger than linear layers. The total number of FLOP of the model increased by about 33x.